### 0. Adjustment of project

In [ ]:
import meep as mp
import conf
import torch
from lib import torch_functions as tf
from lib import numpy_functions as nf
from lib import core_finctions as cf
from lib import my_gui, my_decoders, my_sources
from lib import help_functions
from autograd import numpy as npa
import numpy as np
mp.verbosity(0)

In [ ]:
"""
objective function
"""

def J(FF):
    """
    FF: an array of shapes (n_points, n_waves, n_fields). 
    penalty_coef: penalty coefficient (selected empirically).
    """
    # calculate the point‑by-point intensity: shape = (n_points, n_waves)
    I = npa.abs(FF[0, :, 2])**2 / mean_IPlane_angle_stack[global_index_angle] / max_IAiry_in_angle_stack[global_index_angle]

    # the basic goal is the wave average
    J_base = npa.mean(I)
    
    # penalty for the spread between waves (variance or std^2)
    penalty = penalty_coef * (npa.std(I))**2 

    return J_base - penalty

class OptFunction(torch.autograd.Function):
    @staticmethod
    def forward(ctx, *x_tensors):

        # Forward pass for both single- and multi-frequency optimization.
        # Unwraps torch tensors to numpy arrays, calls external opt(), and saves gradients.

        rho_list = []
        shapes = []
        # Extract data and shapes
        for x in x_tensors:
            arr = x.detach().cpu().numpy()
            # if extra leading dim, squeeze it
            if arr.ndim > 1 and arr.shape[0] == 1:
                arr = arr.squeeze(0)
            shapes.append(arr.shape)
            rho_list.append(arr.flatten())

        # Call external optimizer: returns objective and gradients
        f0, grad_rho = opt(rho_list)

        # Save context for backward
        ctx.grad_rho = grad_rho
        ctx.shapes = shapes
        ctx.device = x_tensors[0].device
        ctx.dtype = x_tensors[0].dtype
        
        # print(f"shapes len = {len(shapes)}")
        # print("shapes:", ctx.shapes)
        # print("returned grad shapes:", [g.shape for g in ctx.grad_rho])
        missing = set(range(len(ctx.shapes))) - set(range(len(ctx.grad_rho)))
        print("Missing gradient indices:", missing)

        # Return scalar objective as torch tensor
        return torch.as_tensor(f0, device=ctx.device, dtype=ctx.dtype)

    @staticmethod
    def backward(ctx, grad_output):
        # Backward pass: reshape and convert numpy gradients to torch,
        # handling both single- and multi-frequency cases.

        grads = []
        for g_np, shape in zip(ctx.grad_rho, ctx.shapes):
            # For multi-frequency, collapse along freq axis
            if g_np.ndim == 2:
                # average gradients over frequencies
                g_arr = np.mean(g_np, axis=1).reshape(shape)
            else:
                # single-frequency gradient
                g_arr = np.asarray(g_np, dtype=np.float64).reshape(shape)

            # Convert to torch tensor and scale by upstream grad_output
            g_torch = torch.from_numpy(g_arr).to(device=ctx.device, dtype=ctx.dtype)
            grads.append(g_torch * grad_output)
    
        # Add zeros for "lost" inputs
        n_missing = len(ctx.shapes) - len(grads)
        if n_missing > 0:
            for idx in range(n_missing):
                shape = ctx.shapes[len(grads) + idx]
                zero = torch.zeros(shape, device=ctx.device, dtype=ctx.dtype)
                grads.append(zero)
            
        # Return gradient for each input
        return tuple(grads)

def opt_torch(*x_tensors):
    # Apply the OptFunction in a torch-friendly way.
    return OptFunction.apply(*x_tensors)

## Main program
### 1.1 Normalization and comparison calculations

In [ ]:
"""
Base parameters
"""
# 1. Shape
global_index_angle = None
rt_tf = tf.calculate_radii_step(D=conf.aperture, orders=conf.orders)
rt_np = nf.calculate_radii_step(D=conf.aperture, orders=conf.orders)
d_rt_np, mid_rt_np = nf.get_drt_midrt(rt_np, conf.orders, is_global = True)

a_initial_params = 1.0
h_initial_params = 1.0

global_a_params = torch.full((conf.orders,), a_initial_params, dtype=torch.float64)
global_h_params = torch.full((conf.orders,), h_initial_params, dtype=torch.float64)

gen_resolution = nf.AdaptiveResolution(
    drt=d_rt_np,
    x_dim=conf.x_dim,
    h_dim= conf.h_dim,
)
resolutions = gen_resolution.generate()
angle_in_rad = np.radians(conf.angle_list)
angle_in_deg = conf.angle_list
angle_after_refraction_in_radii = my_sources.refraction_input(angle_in_deg, conf.in_epsilon)
angle_after_refraction_in_deg = np.rad2deg(angle_after_refraction_in_radii)

# 2. Sources
    # A complete set of sources for all n_angels corners.
    # src is the only plane wave with one long wave
    # batch is a group of continuous plane waves with different wavelengths at the same angle
    # stack is a group with different wave angles of a group of waves with different wavelengths
    
stack_batch_src = []
for angle in angle_after_refraction_in_radii:
    src = my_sources.BatchEigenmodeSource(
        frequencies=conf.inc_frequents_array,
        resolution=conf.sim_resolution,
        diaphragm=conf.diaphragm,
        center=mp.Vector3(x=conf.main_source_center.x, y = 0 ),
        angle_in_rad=angle,
        epsilon=conf.in_epsilon,
        verbose=False,
    )
    batch_src = src.get_sources()
    stack_batch_src.append(batch_src)

# 3. Optimizer
num_steps = 50       # number of optimization iterations
lr=0.3               # the learning rate for Adam.
beta = 2.0           # smoothing parameter for get_weights
penalty_coef = 0.0


    # 3.1 Initializing the parameters
p_min, p_max = 0.01, 50.0
h_min, h_max = 0.01, conf.at_hei

    # 3.2. start the decoders for the reverse transition to [p_min, p_max] and [h_min, h_max]
decoder_a = my_decoders.SigmoidDecoder(pmin=p_min, pmax=p_max)
decoder_h = my_decoders.SigmoidDecoder(pmin=h_min, pmax=h_max)

# 4. Local monitor
locNearRegion = mp.Near2FarRegion(center=conf.flux_center, size=mp.Vector3(x = 0, y=conf.loc_cell.y)) 

# 5. GUI output
panel = my_gui.LiveInfoPanel()
panel.show()

# 6. Save and load
# "name" if you load, None if you save

filename_focus_IAiry = "focus_Airy_simulation_D=2000.0_f=2000.0_diaph=2000_res=10.npy"
filename_base_IAiry = "base_Airy_simulation_D=2000.0_f=2000.0_diaph=2000_res=10.npy"
file_name_IBase = "base_IPlane_D=2000.0_f=2000.0_diaph=2000_res=10.npy"

"""
Examples

# filename_focus_IAiry = "focus_Airy_simulation_D=6000.0_f=2000.0_diaph=2000_res=10.npy"
# filename_base_IAiry = "base_Airy_simulation_D=6000.0_f=2000.0_diaph=2000_res=10.npy"
# file_name_IBase = "base_IPlane_D=6000.0_f=2000.0_diaph=2000_res=10.npy"

# filename_focus_IAiry = None
# filename_base_IAiry = None
# file_name_IBase = None

"""

In [ ]:
IAiry_norm_scaled_in_angle_stack = []
max_IAiry_in_angle_stack = []
IPlane_base_angle_stack = []
mean_IPlane_angle_stack = []

# Airy wave
IAiry, IBaseAiry, sim = nf.get_IAiry_simulation(
    monitor =  locNearRegion, 
    cell = conf.loc_cell, 
    filename_focus_airy = filename_focus_IAiry, 
    filename_base_airy= filename_base_IAiry, 
    resolution = conf.sim_resolution,
    get_base=False,
    symmetries=[mp.Mirror(mp.Y)])
abscissa = nf.get_abscissa(conf.spot_ff, IAiry/IAiry.max(axis = 0), 0, 0)
airy_efficiency_r = nf.get_focusing_efficiency(IAiry/IAiry.max(axis = 0), abscissa = abscissa)
nf.airy_characteristics()
print(f"airy_efficiency_r: {airy_efficiency_r}")
for i, (angle_rad, angle_deg) in enumerate(zip(angle_in_rad, angle_in_deg)):
    abscissa = nf.get_abscissa(conf.spot_ff, IAiry/IAiry.max(axis = 0), angle_in_deg[i], i * conf.diaphragm)
    IAiry_norm_in_angle = IAiry * np.cos(angle_rad)
    max_IAiry_in_angle = IAiry_norm_in_angle.max(axis=0)
    max_IAiry_in_angle_stack.append(max_IAiry_in_angle)
    IAiry_norm_scaled_in_angle = IAiry_norm_in_angle / max_IAiry_in_angle
    IAiry_norm_scaled_in_angle_stack.append(IAiry_norm_scaled_in_angle)
    nf.get_plot_intensity(
        IAiry_norm_scaled_in_angle,
        out_angle_in_deg=angle_deg,
        wavelength_array=conf.inc_wavelength_array,
        abscissa=abscissa,
        isfwhm=True,
    )

# IBase for normalization
IPlane_base_angle_stack = nf.get_Ibase_batch_simulation(
    monitor =  locNearRegion,
    sources_batch=stack_batch_src, 
    cell=conf.loc_cell, 
    frequents_array=conf.inc_frequents_array,
    filename= file_name_IBase, 
    symmetries = [mp.Mirror(mp.Y)])

mean_IPlane_angle_stack = np.mean(IPlane_base_angle_stack, axis=2) # include average I_base for every angles

In [ ]:
# global_a_params = torch.load("global_a_params.pt", weights_only=True)
# global_h_params = torch.load("global_h_params.pt", weights_only=True)

NOTE:
If we are uploading data, then the next step is not necessary.

### 1.2 Initializing the optimizer

In [ ]:
# # ____ Multi-angels _____
# opt_zone_start = 0
# opt_zone_end = conf.n_angles
# 
# for i in range(opt_zone_start, opt_zone_end):
#     global_index_angle = i
#     if angle_in_deg[i] == 0:
#         penalty_coef = 5.0
#         lr = 0.5
#     if angle_in_deg[i] > 10:
#         penalty_coef = 5.0
#         lr = 0.25
#     if angle_in_deg[i] > 20:
#         penalty_coef = 0.0
#         lr = 0.1
#         
#     # 0.1) Initializing the optimizer
#     opt = cf.init_loc_opt(i, conf.loc_orders,J,locNearRegion=locNearRegion,stack_batch_src=stack_batch_src, resolutions = resolutions, a_initial=a_initial_params, h_initial=h_initial_params,rt_np = rt_np,is_plot = True)
#     print(f"angle {i}: Initializing DONE")
#     
#     # 0.2) Initial hyper params
#     loc_optimizer, store_loc_history, loc_u_a, loc_u_h = cf.init_loc_opt_hyper(i, conf.loc_orders, a_initial=a_initial_params, h_initial=h_initial_params, p_min=p_min, p_max=p_max, h_min=h_min, h_max=h_max, lr=lr, regime=1.5)
#     
#     # 1) Begin optimization loop
#     loc_history, loc_best_a_params, loc_best_h_params  = cf.run_loc_opt(opt_torch, i, conf.loc_orders, loc_optimizer, loc_u_a, loc_u_h, store_loc_history, beta, num_steps, decoder_a, decoder_h, resolutions, regime=1.5)
#     
#     # 2) Visualise 
#     cf.viz_loc_opt(i, loc_history, loc_best_a_params,loc_best_h_params, conf.loc_orders, is_save_params=False, save_path="opt_progress_and_param.pdf", save_format="pdf")
#     
#     # 3) Update global_params
#     cf.update_global_params(i, global_a_params, global_h_params, loc_best_a_params, loc_best_h_params, is_save = True)
#     
#     # 4) Save history
#     help_functions.save_history(loc_history, save=True, index=global_index_angle)

In [ ]:
# ____ Single-angel _____
i = 0
global_index_angle = i
lr = 0.2
# Инициируем симмуляцию с оптимизатором
opt = cf.init_loc_opt(i,2*conf.orders,J,locNearRegion=locNearRegion,stack_batch_src=stack_batch_src, resolutions = resolutions, a_initial=a_initial_params, h_initial=h_initial_params,rt_np = rt_np,is_plot = False)

# 0) Initial hyper params   
loc_optimizer, store_loc_history, loc_u_a, loc_u_h = cf.init_loc_opt_hyper(i, 2*conf.orders, a_initial=a_initial_params, h_initial=h_initial_params, p_min=p_min, p_max=p_max, h_min=h_min, h_max=h_max, lr=lr, regime=1.5)

# 1) Begin optimization loop
loc_history, loc_best_a_params, loc_best_h_params  = cf.run_loc_opt(opt_torch, i, 2*conf.orders, loc_optimizer, loc_u_a, loc_u_h, store_loc_history, beta, num_steps, decoder_a, decoder_h, resolutions, regime=1.5)

# 2) Visualise 
cf.viz_loc_opt(i, loc_history, loc_best_a_params,loc_best_h_params, conf.loc_orders, is_save_params=False, save_path="opt_progress_and_param.pdf", save_format="pdf")

# 3) Update global_params
cf.update_global_params(i, global_a_params, global_h_params, loc_best_a_params, loc_best_h_params, is_save = True)

# 4) Save history
help_functions.save_history(loc_history, save=True, index = 10)
# help_functions.save_history(loc_history, save=True, index=global_index_angle)

### 3. TEST

In [ ]:
# # ____ Multi-angle ____
# opt_zone_start = 0
# opt_zone_end = conf.n_angles
# for i in range(opt_zone_start, opt_zone_end):
#     test_loc_a = cf.get_local_tensor_from_global(global_a_params, i, conf.loc_orders)
#     test_loc_h = cf.get_local_tensor_from_global(global_h_params, i, conf.loc_orders)
#     print(f"tested region: {i}, angle {angle_in_deg[i]} includes elements: {len(global_a_params)}")
#     IOptimized = cf.test_loc(i, conf.loc_orders, stack_batch_src[i],rt_np, resolutions, locNearRegion, test_loc_a, test_loc_h, is_plot_sim = True)
#     max_per_column = IOptimized.max(axis=0)
#     IOptimized_scaled = IOptimized / max_per_column
#     # abscissa = nf.get_abscissa(conf.spot_ff, IOptimized_scaled[:,0], angle_in_deg[i], i * conf.diaphragm)
#     abscissa = nf.get_abscissa(conf.spot_ff, IOptimized_scaled, angle_in_deg[i], i * conf.diaphragm)
#     
#     I_crop, absc_crop = help_functions.recenter_and_crop(IOptimized_scaled, abscissa, ref_index=1)
# 
#     # Outputs data
#     nf.get_plot_intensity(
#         I_crop,
#         out_angle_in_deg=angle_in_deg[i],
#         wavelength_array=conf.inc_wavelength_array, 
#         abscissa=absc_crop,
#         isfwhm=False,
#     )
#     nf.plot_psf(
#         conf.inc_nfreq, conf.spot_ff,
#         I_crop,
#         abscissa=absc_crop
#     )
#     nf.plot_mtf(
#         conf.spot_ff, I_crop,
#         max_view=150,
#         in_angle_in_deg=angle_after_refraction_in_deg[i],
#         out_angle_in_deg=angle_in_deg[i],
#         dif_lim=True, 
#         smooth=True,
#         I_mtf_theory = None,
#     )
#     opt_efficiency_r = nf.get_focusing_efficiency(
#         I_crop/I_crop.max(axis = 0),
#         abscissa=absc_crop,
#         )
# 
#     kpd = opt_efficiency_r/airy_efficiency_r
#     mean_val = np.mean(kpd)      
#     var_val = np.var(kpd)       
#     std_val = np.std(kpd)
# 
#     # Logs
#     print(f"opt_efficiency_r: {opt_efficiency_r}")
#     print(f"kpd: {kpd}")
#     print("Average:", mean_val)
#     print("Variance:", var_val)
#     print("St. deviation:", std_val)

In [ ]:
# ____ Single angle ____
opt_zone_start = 0
i = opt_zone_start
test_loc_a = cf.get_local_tensor_from_global(global_a_params, i, conf.loc_orders)
test_loc_h = cf.get_local_tensor_from_global(global_h_params, i, conf.loc_orders)
print(f"tested region: {i}, angle {angle_in_deg[i]} includes elements: {len(test_loc_a)}")
IOptimized = cf.test_loc(i, conf.loc_orders, stack_batch_src[i],rt_np, resolutions, locNearRegion, test_loc_a, test_loc_h, is_plot_sim = True)
max_per_column = IOptimized.max(axis=0)
IOptimized_scaled = IOptimized / max_per_column
abscissa = nf.get_abscissa(conf.spot_ff, IOptimized_scaled[:,0], angle_in_deg[i], i * conf.diaphragm)

# Crop if needed
# I_crop, absc_crop = help_functions.recenter_and_crop(IOptimized_scaled, abscissa, ref_index=0)
I_crop = IOptimized_scaled
absc_crop = abscissa

# Outputs data
nf.get_plot_intensity(
    I_crop,
    out_angle_in_deg=angle_in_deg[i],
    wavelength_array=conf.inc_wavelength_array, 
    abscissa=absc_crop,
    isfwhm=True,
)
nf.plot_psf(
    conf.inc_nfreq, conf.spot_ff,
    I_crop,
    abscissa=absc_crop,
    save_path = "psf_plot.pdf", 
    save_format="pdf",
)

nf.plot_mtf(
    conf.spot_ff, I_crop,
    max_view=150,
    in_angle_in_deg=angle_after_refraction_in_deg[i],
    out_angle_in_deg=angle_in_deg[i],
    dif_lim=True, 
    smooth=True,
    I_mtf_theory = None,
    save_path="mtf_plot.pdf",
    save_format="pdf"
)

opt_efficiency_r = nf.get_focusing_efficiency(
    I_crop/I_crop.max(axis = 0),
    abscissa=absc_crop,
    )

kpd = opt_efficiency_r/airy_efficiency_r
mean_val = np.mean(kpd)      
var_val = np.var(kpd)       
std_val = np.std(kpd)

# Logs
print(f"opt_efficiency_r: {opt_efficiency_r}")
print(f"kpd: {kpd}")
print("Average:", mean_val)
print("Variance:", var_val)
print("St. deviation:", std_val)